# 06 — CUI TF-IDF Similarity

Goal:

1. Use existing `cui_set` files generated from `Title + Summary`.
2. Weight CUIs by TF-IDF so rare CUIs matter more.
3. Find top similar pairs:
   - human × mouse
   - within human
   - within mouse
4. Add metadata/source flags.
5. Remove exact duplicate CUI/summary pairs.
6. Compare CUI similarity with raw text similarity.

In [2]:
from pathlib import Path
import ast
import heapq
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

python(29160) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


## 1. Configuration

In [3]:
# Input files
HUMAN_CUI_PATH = Path("metadata/human_with_cuis.pkl")
MOUSE_CUI_PATH = Path("metadata/mouse_with_cuis.pkl")

# Optional source metadata files generated by src/find_pmid.py
HUMAN_SOURCE_PATH = Path("metadata/gse_pmid_subseries_of_human.tsv")
MOUSE_SOURCE_PATH = Path("metadata/gse_pmid_subseries_of_mouse.tsv")

# Output directory
RESULT_DIR = Path("similarity-results")
RESULT_DIR.mkdir(exist_ok=True)

# Similarity settings
TOP_N = 1000
CHUNK_SIZE = 100

# Output files
HUMAN_MOUSE_OUT = RESULT_DIR / "top_1000_human_mouse_cui_tfidf_pairs.csv"
WITHIN_HUMAN_OUT = RESULT_DIR / "top_1000_within_human_cui_tfidf_pairs.csv"
WITHIN_MOUSE_OUT = RESULT_DIR / "top_1000_within_mouse_cui_tfidf_pairs.csv"

ALL_PAIRS_OUT = RESULT_DIR / "top_all_cui_tfidf_similarity_pairs.csv"
WITH_FLAGS_OUT = RESULT_DIR / "top_all_cui_tfidf_pairs_with_metadata_source_flags.csv"
EXCLUDED_EXACT_OUT = RESULT_DIR / "excluded_exact_cui_or_summary_pairs.csv"
FILTERED_EXACT_OUT = RESULT_DIR / "top_all_cui_tfidf_pairs_filtered_exact.csv"
TEXT_SIM_OUT = RESULT_DIR / "top_all_cui_tfidf_pairs_with_text_similarity.csv"

## 2. Helper functions

In [4]:
def ensure_cui_set(x):
    """Return a Python set for CUI data stored as set/list/string."""
    if isinstance(x, set):
        return x
    if isinstance(x, list):
        return set(x)
    if isinstance(x, str):
        try:
            return set(ast.literal_eval(x))
        except Exception:
            return set()
    return set()


def normalize_text(x, remove_punctuation=False):
    """Normalize text for exact/near-exact comparison."""
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    if remove_punctuation:
        x = re.sub(r"[^\w\s]", "", x)
    return x.strip()


def split_source_field(x):
    """Split source metadata fields such as PMID/SRA/BioProject into comparable sets."""
    if pd.isna(x):
        return set()
    if isinstance(x, list):
        return set(map(str, x))
    return set(i.strip() for i in str(x).split(";") if i.strip())


def shared_cuis(cui_set_1, cui_set_2):
    return sorted(cui_set_1 & cui_set_2)


def shared_idf_sum(cuis, idf_lookup):
    return sum(idf_lookup.get(cui, 0) for cui in cuis)

## 3. Load existing CUI sets

In [5]:
human_df = pd.read_pickle(HUMAN_CUI_PATH)
mouse_df = pd.read_pickle(MOUSE_CUI_PATH)

human_df["cui_set"] = human_df["cui_set"].apply(ensure_cui_set)
mouse_df["cui_set"] = mouse_df["cui_set"].apply(ensure_cui_set)

print("Human GSEs:", len(human_df))
print("Mouse GSEs:", len(mouse_df))
print("Human columns:", list(human_df.columns))
print("Mouse columns:", list(mouse_df.columns))

Human GSEs: 3395
Mouse GSEs: 4066
Human columns: ['gse_id', 'species', 'title', 'summary', 'text', 'cui_set']
Mouse columns: ['gse_id', 'species', 'title', 'summary', 'text', 'cui_set']


## 4. Build CUI TF-IDF matrices

In [6]:
human_df["cui_doc"] = human_df["cui_set"].apply(lambda s: " ".join(sorted(s)))
mouse_df["cui_doc"] = mouse_df["cui_set"].apply(lambda s: " ".join(sorted(s)))

all_cui_docs = pd.concat(
    [human_df["cui_doc"], mouse_df["cui_doc"]],
    ignore_index=True
)

cui_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    binary=True,
    use_idf=True,
    smooth_idf=True,
    norm="l2"
)

X_all = cui_vectorizer.fit_transform(all_cui_docs)

n_human = len(human_df)
X_human = X_all[:n_human]
X_mouse = X_all[n_human:]

cui_idf = dict(zip(cui_vectorizer.get_feature_names_out(), cui_vectorizer.idf_))

print("Human TF-IDF matrix:", X_human.shape)
print("Mouse TF-IDF matrix:", X_mouse.shape)
print("Unique CUIs:", len(cui_idf))

Human TF-IDF matrix: (3395, 33968)
Mouse TF-IDF matrix: (4066, 33968)
Unique CUIs: 33968


## 5. Find top CUI-TF-IDF pairs

In [7]:
def compute_top_cui_tfidf_pairs(
    left_df,
    right_df,
    X_left,
    X_right,
    comparison_label,
    output_path,
    top_n=1000,
    chunk_size=100,
    within=False
):
    """
    Compute top CUI-TF-IDF cosine similarity pairs.

    For within-species comparisons, set within=True to remove:
    - self pairs
    - duplicated reverse pairs
    """

    if output_path.exists():
        result = pd.read_csv(output_path)
        print(f"Loaded existing file: {output_path}")
        return result

    heap = []
    counter = 0

    for start in tqdm(range(0, X_left.shape[0], chunk_size), desc=comparison_label):
        end = min(start + chunk_size, X_left.shape[0])

        sim_chunk = (X_left[start:end] @ X_right.T).toarray()

        if within:
            for local_i in range(sim_chunk.shape[0]):
                global_i = start + local_i
                sim_chunk[local_i, :global_i + 1] = 0

        flat = sim_chunk.ravel()
        candidate_n = min(top_n, flat.size)
        candidate_idx = np.argpartition(flat, -candidate_n)[-candidate_n:]

        for idx in candidate_idx:
            local_i, j = np.unravel_index(idx, sim_chunk.shape)
            i = start + local_i
            score = sim_chunk[local_i, j]

            if score <= 0:
                continue

            left_cuis = left_df.iloc[i]["cui_set"]
            right_cuis = right_df.iloc[j]["cui_set"]
            shared = shared_cuis(left_cuis, right_cuis)

            row = {
                "comparison": comparison_label,
                "gse_1": left_df.iloc[i]["gse_id"],
                "gse_2": right_df.iloc[j]["gse_id"],
                "tfidf_cosine": score,
                "shared_cui_count": len(shared),
                "shared_idf_sum": shared_idf_sum(shared, cui_idf),
                "shared_cuis": shared,
                "title_1": left_df.iloc[i]["title"],
                "title_2": right_df.iloc[j]["title"],
            }

            item = (score, row["shared_idf_sum"], row["shared_cui_count"], counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    result = (
        pd.DataFrame([item[-1] for item in heap])
        .sort_values(["tfidf_cosine", "shared_idf_sum", "shared_cui_count"], ascending=False)
        .reset_index(drop=True)
    )

    result.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

    return result

In [8]:
top_human_mouse = compute_top_cui_tfidf_pairs(
    left_df=human_df,
    right_df=mouse_df,
    X_left=X_human,
    X_right=X_mouse,
    comparison_label="human_mouse",
    output_path=HUMAN_MOUSE_OUT,
    top_n=TOP_N,
    chunk_size=CHUNK_SIZE,
    within=False
)

top_within_human = compute_top_cui_tfidf_pairs(
    left_df=human_df,
    right_df=human_df,
    X_left=X_human,
    X_right=X_human,
    comparison_label="within_human",
    output_path=WITHIN_HUMAN_OUT,
    top_n=TOP_N,
    chunk_size=CHUNK_SIZE,
    within=True
)

top_within_mouse = compute_top_cui_tfidf_pairs(
    left_df=mouse_df,
    right_df=mouse_df,
    X_left=X_mouse,
    X_right=X_mouse,
    comparison_label="within_mouse",
    output_path=WITHIN_MOUSE_OUT,
    top_n=TOP_N,
    chunk_size=CHUNK_SIZE,
    within=True
)

Loaded existing file: similarity-results/top_1000_human_mouse_cui_tfidf_pairs.csv
Loaded existing file: similarity-results/top_1000_within_human_cui_tfidf_pairs.csv
Loaded existing file: similarity-results/top_1000_within_mouse_cui_tfidf_pairs.csv


## 6. Combine all top pairs

In [9]:
all_pairs = pd.concat(
    [top_human_mouse, top_within_human, top_within_mouse],
    ignore_index=True
)

all_pairs = (
    all_pairs
    .sort_values(["tfidf_cosine", "shared_idf_sum", "shared_cui_count"], ascending=False)
    .drop_duplicates(subset=["comparison", "gse_1", "gse_2"])
    .reset_index(drop=True)
)

all_pairs.to_csv(ALL_PAIRS_OUT, index=False)

print("All top pairs:", len(all_pairs))
all_pairs.head(10)

All top pairs: 3000


,comparison,gse_1,gse_2,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2
0,within_mouse,GSE101623,GSE101624,1.0,165,965.694778,"['C0001554', 'C0001563', 'C0005884', 'C0014457...",The neuropeptide Neuromedin U stimulates innat...,The neuropeptide Neuromedin U stimulates innat...
1,within_mouse,GSE76864,GSE76865,1.0,109,622.012226,"['C0001688', 'C0003241', 'C0003242', 'C0004561...",Independent roles of switching and hypermutati...,Independent roles of switching and hypermutati...
2,within_mouse,GSE73559,GSE73560,1.0,110,565.120041,"['C0004561', 'C0007634', 'C0011155', 'C0011377...",Gene expression analysis to identify Klf2 targ...,Gene expression analysis to identify Klf2 targ...
3,within_human,GSE94528,GSE94999,1.0,109,564.937214,"['C0001272', 'C0001688', 'C0004561', 'C0006826...","H3B-8800, a novel oral splicing modulator, ind...","H3B-8800, a novel oral splicing modulator, ind..."
4,within_human,GSE81074,GSE81080,1.0,108,547.524853,"['C0002793', 'C0002874', 'C0004561', 'C0005839...",Differentiation of human embryonic stem cells ...,Differentiation of human embryonic stem cells ...
5,within_mouse,GSE77736,GSE77740,1.0,103,539.182528,"['C0001272', 'C0001779', 'C0001811', 'C0004561...",Single Novel single cell assay reveals progres...,Single Novel single cell assay reveals progres...
6,human_mouse,GSE103658,GSE103725,1.0,101,496.886869,"['C0001792', 'C0002976', 'C0015609', 'C0017262...",Expression changes in Melanomas pre MAPKi trea...,Expression changes in Melanomas pre MAPKi trea...
7,within_mouse,GSE103185,GSE104325,1.0,85,417.046933,"['C0001779', 'C0004909', 'C0006104', 'C0007613...",Early-life gene expression in neurons modulate...,Early-life gene expression in neurons modulate...
8,within_human,GSE96562,GSE96563,1.0,149,773.952612,"['C0003320', 'C0003334', 'C0003341', 'C0004561...",Single cell RNA-seq reveals expansion of IGRP-...,Single cell RNA-seq reveals expansion of IGRP-...
9,within_human,GSE81497,GSE81498,1.0,133,657.067069,"['C0002684', 'C0003015', 'C0005495', 'C0006826...",Multiple mechanisms disrupt let-7 miRNA biogen...,Multiple mechanisms disrupt let-7 miRNA biogen...


## 7. Add metadata and source flags

In [10]:
# Metadata lookup from existing human/mouse data
human_lookup = human_df.set_index("gse_id").to_dict(orient="index")
mouse_lookup = mouse_df.set_index("gse_id").to_dict(orient="index")
metadata_lookup = {**human_lookup, **mouse_lookup}


def get_metadata(gse, field):
    return metadata_lookup.get(gse, {}).get(field, np.nan)


def get_cui_set(gse):
    return ensure_cui_set(get_metadata(gse, "cui_set"))


pairs_with_flags = all_pairs.copy()

pairs_with_flags["title_1_full"] = pairs_with_flags["gse_1"].apply(lambda g: get_metadata(g, "title"))
pairs_with_flags["title_2_full"] = pairs_with_flags["gse_2"].apply(lambda g: get_metadata(g, "title"))
pairs_with_flags["summary_1"] = pairs_with_flags["gse_1"].apply(lambda g: get_metadata(g, "summary"))
pairs_with_flags["summary_2"] = pairs_with_flags["gse_2"].apply(lambda g: get_metadata(g, "summary"))

pairs_with_flags["summary_1_norm"] = pairs_with_flags["summary_1"].apply(lambda x: normalize_text(x, remove_punctuation=True))
pairs_with_flags["summary_2_norm"] = pairs_with_flags["summary_2"].apply(lambda x: normalize_text(x, remove_punctuation=True))

pairs_with_flags["same_summary"] = pairs_with_flags["summary_1_norm"] == pairs_with_flags["summary_2_norm"]
pairs_with_flags["same_cui_set"] = pairs_with_flags.apply(
    lambda r: get_cui_set(r["gse_1"]) == get_cui_set(r["gse_2"]),
    axis=1
)

print("Exact same summary pairs:", pairs_with_flags["same_summary"].sum())
print("Exact same CUI-set pairs:", pairs_with_flags["same_cui_set"].sum())

Exact same summary pairs: 452
Exact same CUI-set pairs: 156


In [11]:
# Load source metadata if available
if HUMAN_SOURCE_PATH.exists() and MOUSE_SOURCE_PATH.exists():
    human_source = pd.read_csv(HUMAN_SOURCE_PATH, sep="\t")
    mouse_source = pd.read_csv(MOUSE_SOURCE_PATH, sep="\t")

    source_df = pd.concat([human_source, mouse_source], ignore_index=True)
    source_lookup = source_df.set_index("gse").to_dict(orient="index")

    print("Loaded source metadata:")
    print("Human source rows:", len(human_source))
    print("Mouse source rows:", len(mouse_source))
else:
    source_lookup = {}
    print("Source metadata files not found.")
    print("Expected files:")
    print(HUMAN_SOURCE_PATH)
    print(MOUSE_SOURCE_PATH)


def get_source(gse, field):
    return source_lookup.get(gse, {}).get(field, np.nan)


def source_overlap(gse_1, gse_2, field):
    values_1 = split_source_field(get_source(gse_1, field))
    values_2 = split_source_field(get_source(gse_2, field))
    return sorted(values_1 & values_2)


source_fields = ["pmid", "subseries", "superseries", "affiliation", "BioProject", "SRA"]

for field in source_fields:
    pairs_with_flags[f"{field}_1"] = pairs_with_flags["gse_1"].apply(lambda g: get_source(g, field))
    pairs_with_flags[f"{field}_2"] = pairs_with_flags["gse_2"].apply(lambda g: get_source(g, field))
    pairs_with_flags[f"shared_{field}"] = pairs_with_flags.apply(
        lambda r: source_overlap(r["gse_1"], r["gse_2"], field),
        axis=1
    )
    pairs_with_flags[f"same_{field}"] = pairs_with_flags[f"shared_{field}"].apply(lambda x: len(x) > 0)


def same_source_label(row):
    if row["same_pmid"] and row["same_subseries"]:
        return pd.Series({"same_source_prediction": True, "same_source_confidence": "Very High"})
    if row["same_pmid"] or row["same_subseries"]:
        return pd.Series({"same_source_prediction": True, "same_source_confidence": "High"})
    if row["same_SRA"] or row["same_BioProject"]:
        return pd.Series({"same_source_prediction": True, "same_source_confidence": "Moderate"})
    return pd.Series({"same_source_prediction": False, "same_source_confidence": "Unknown"})


pairs_with_flags[["same_source_prediction", "same_source_confidence"]] = pairs_with_flags.apply(
    same_source_label,
    axis=1
)

pairs_with_flags["same_source_confidence"].value_counts(dropna=False)

Loaded source metadata:
Human source rows: 3395
Mouse source rows: 4064


same_source_confidence
Unknown      2237
Very High     578
High          185
Name: count, dtype: int64

## 8. Remove exact CUI-set and exact summary pairs

In [12]:
excluded_exact = pairs_with_flags[
    pairs_with_flags["same_cui_set"] | pairs_with_flags["same_summary"]
].copy()

filtered_pairs = pairs_with_flags[
    ~(pairs_with_flags["same_cui_set"] | pairs_with_flags["same_summary"])
].copy().reset_index(drop=True)

pairs_with_flags.to_csv(WITH_FLAGS_OUT, index=False)
excluded_exact.to_csv(EXCLUDED_EXACT_OUT, index=False)
filtered_pairs.to_csv(FILTERED_EXACT_OUT, index=False)

print("Pairs before filtering:", len(pairs_with_flags))
print("Excluded exact CUI-set or summary pairs:", len(excluded_exact))
print("Pairs after filtering:", len(filtered_pairs))
print("Saved:", FILTERED_EXACT_OUT)

# filtered_pairs.head(10)

Pairs before filtering: 3000
Excluded exact CUI-set or summary pairs: 470
Pairs after filtering: 2530
Saved: similarity-results/top_all_cui_tfidf_pairs_filtered_exact.csv


## 9. Compare raw text similarity with CUI-TF-IDF similarity

In [13]:
# ------------------------------------------------------------
# 9. Compare edit distance with CUI-TF-IDF similarity
# ------------------------------------------------------------

analysis_pairs = filtered_pairs.copy()
analysis_pairs = analysis_pairs.rename(columns={"tfidf_cosine": "cui_tfidf_cosine"})

# normalize
def normalize_text_for_distance(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()


analysis_pairs["title_1_norm"] = analysis_pairs["title_1_full"].apply(normalize_text_for_distance)
analysis_pairs["title_2_norm"] = analysis_pairs["title_2_full"].apply(normalize_text_for_distance)

analysis_pairs["summary_1_norm"] = analysis_pairs["summary_1"].apply(normalize_text_for_distance)
analysis_pairs["summary_2_norm"] = analysis_pairs["summary_2"].apply(normalize_text_for_distance)

analysis_pairs["text_1_norm"] = (
    analysis_pairs["title_1_norm"] + " " + analysis_pairs["summary_1_norm"]
).str.strip()

analysis_pairs["text_2_norm"] = (
    analysis_pairs["title_2_norm"] + " " + analysis_pairs["summary_2_norm"]
).str.strip()

In [14]:
# calculate similarity
try:
    from rapidfuzz.distance import DamerauLevenshtein
except ImportError:
    raise ImportError("Please install rapidfuzz first: pip install rapidfuzz")


def damerau_similarity(text_1, text_2):
    text_1 = normalize_text_for_distance(text_1)
    text_2 = normalize_text_for_distance(text_2)

    max_len = max(len(text_1), len(text_2))

    if max_len == 0:
        return 0

    distance = DamerauLevenshtein.distance(text_1, text_2)

    return 1 - distance / max_len


analysis_pairs["title_damerau_sim"] = analysis_pairs.apply(
    lambda r: damerau_similarity(r["title_1_norm"], r["title_2_norm"]),
    axis=1
)

analysis_pairs["summary_damerau_sim"] = analysis_pairs.apply(
    lambda r: damerau_similarity(r["summary_1_norm"], r["summary_2_norm"]),
    axis=1
)

analysis_pairs["text_damerau_sim"] = analysis_pairs.apply(
    lambda r: damerau_similarity(r["text_1_norm"], r["text_2_norm"]),
    axis=1
)

In [15]:
# define edit_high and cui_high
# Edit-distance thresholds
TITLE_EDIT_HIGH_THRESHOLD = 0.90
SUMMARY_EDIT_HIGH_THRESHOLD = 0.95
TEXT_EDIT_HIGH_THRESHOLD = 0.90

# CUI high = top 25% among current candidate pairs
CUI_HIGH_QUANTILE = 0.75

cui_high_cutoff = analysis_pairs["cui_tfidf_cosine"].quantile(CUI_HIGH_QUANTILE)

analysis_pairs["title_edit_high"] = (
    analysis_pairs["title_damerau_sim"] >= TITLE_EDIT_HIGH_THRESHOLD
)

analysis_pairs["summary_edit_high"] = (
    analysis_pairs["summary_damerau_sim"] >= SUMMARY_EDIT_HIGH_THRESHOLD
)

analysis_pairs["text_edit_high"] = (
    analysis_pairs["text_damerau_sim"] >= TEXT_EDIT_HIGH_THRESHOLD
)

analysis_pairs["edit_high"] = (
    analysis_pairs["title_edit_high"] |
    analysis_pairs["summary_edit_high"] |
    analysis_pairs["text_edit_high"]
)

analysis_pairs["cui_high"] = (
    analysis_pairs["cui_tfidf_cosine"] >= cui_high_cutoff
)

print("CUI high cutoff:", cui_high_cutoff)
print("edit_high:", analysis_pairs["edit_high"].sum())
print("cui_high:", analysis_pairs["cui_high"].sum())

CUI high cutoff: 0.4643761906793913
edit_high: 140
cui_high: 633


`edit_high`, `cui_high`
1. edit_high + cui_high: too similar
    - combine with pubmedid recognition, how much is the agreement
2. edit_low + cui_high: similar biological concept but diff text
3. edit_high + cui_low: looks similar but something might be wrong with cui recognition
4. edit_low + cui_low: we think that they are not matching

In [16]:
# turn into four groups
def classify_edit_cui_pattern(row):
    if row["edit_high"] and row["cui_high"]:
        return "edit_high_cui_high_too_similar"

    if (not row["edit_high"]) and row["cui_high"]:
        return "edit_low_cui_high_bio_concept_match"

    if row["edit_high"] and (not row["cui_high"]):
        return "edit_high_cui_low_possible_cui_miss"

    return "edit_low_cui_low_not_matching"


analysis_pairs["edit_cui_pattern"] = analysis_pairs.apply(
    classify_edit_cui_pattern,
    axis=1
)

analysis_pairs["edit_cui_pattern"].value_counts()

edit_cui_pattern
edit_low_cui_low_not_matching          1884
edit_low_cui_high_bio_concept_match     506
edit_high_cui_high_too_similar          127
edit_high_cui_low_possible_cui_miss      13
Name: count, dtype: int64

## 10. Summary tables and outputs

In [17]:
score_columns = [
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim"
]

score_summary = analysis_pairs[score_columns].describe()
score_correlation = analysis_pairs[score_columns].corr()

display(score_summary)
display(score_correlation)

,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim
count,2530.000000,2530.000000,2530.000000,2530.000000
mean,0.437250,0.347982,0.343747,0.354715
std,0.146316,0.204578,0.207199,0.186965
min,0.306077,0.038835,0.033967,0.082949
25%,0.341341,0.217391,0.223847,0.245685
50%,0.384013,0.265976,0.264608,0.282353
75%,0.464376,0.400000,0.377069,0.392335
max,0.999112,1.000000,0.999186,0.999164


,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim
cui_tfidf_cosine,1.000000,0.595830,0.764184,0.796292
title_damerau_sim,0.595830,1.000000,0.564015,0.699008
summary_damerau_sim,0.764184,0.564015,1.000000,0.961619
text_damerau_sim,0.796292,0.699008,0.961619,1.000000


In [18]:
pattern_counts = analysis_pairs["edit_cui_pattern"].value_counts().reset_index()
pattern_counts.columns = ["edit_cui_pattern", "n_pairs"]

display(pattern_counts)

,edit_cui_pattern,n_pairs
0,edit_low_cui_low_not_matching,1884
1,edit_low_cui_high_bio_concept_match,506
2,edit_high_cui_high_too_similar,127
3,edit_high_cui_low_possible_cui_miss,13


In [19]:
# 10.3 Compare edit/CUI pattern with PMID match
if "same_pmid" not in analysis_pairs.columns:
    raise ValueError("same_pmid column not found. Run Section 7 source flagging first.")

pmid_agreement_table = pd.crosstab(
    analysis_pairs["edit_cui_pattern"],
    analysis_pairs["same_pmid"],
    margins=True
)

display(pmid_agreement_table)

pmid_match_rate_by_pattern = (
    analysis_pairs
    .groupby("edit_cui_pattern")["same_pmid"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={
        "count": "n_pairs",
        "sum": "n_same_pmid",
        "mean": "pmid_match_rate"
    })
)

pmid_match_rate_by_pattern["pmid_match_rate"] *= 100

display(pmid_match_rate_by_pattern)

same_pmid,False,True,All
edit_cui_pattern,,,
edit_high_cui_high_too_similar,45,82,127
edit_high_cui_low_possible_cui_miss,3,10,13
edit_low_cui_high_bio_concept_match,347,159,506
edit_low_cui_low_not_matching,1833,51,1884
All,2228,302,2530


,edit_cui_pattern,n_pairs,n_same_pmid,pmid_match_rate
0,edit_high_cui_high_too_similar,127,82,64.566929
1,edit_high_cui_low_possible_cui_miss,13,10,76.923077
2,edit_low_cui_high_bio_concept_match,506,159,31.422925
3,edit_low_cui_low_not_matching,1884,51,2.707006


In [20]:
# edit_high + cui_high vs PMID
too_similar_pairs = analysis_pairs[
    analysis_pairs["edit_cui_pattern"] == "edit_high_cui_high_too_similar"
].copy()

n_too_similar = len(too_similar_pairs)
n_too_similar_same_pmid = too_similar_pairs["same_pmid"].sum()

if n_too_similar > 0:
    pmid_agreement_percent = n_too_similar_same_pmid / n_too_similar * 100
else:
    pmid_agreement_percent = 0

print("edit_high + cui_high pairs:", n_too_similar)
print("also same PMID:", n_too_similar_same_pmid)
print(f"PMID agreement percentage: {pmid_agreement_percent:.2f}%")

edit_high + cui_high pairs: 127
also same PMID: 82
PMID agreement percentage: 64.57%


In [ ]:
# not high-high but PMID
too_similar_not_same_pmid = (
    too_similar_pairs[
        ~too_similar_pairs["same_pmid"]
    ]
    .sort_values("cui_tfidf_cosine", ascending=False)
    .reset_index(drop=True)
)

too_similar_not_same_pmid[[
    "comparison",
    "gse_1",
    "gse_2",
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim",
    "same_pmid",
    "same_source_prediction",
    "same_source_confidence",
    "shared_pmid",
    "title_1_full",
    "title_2_full"
]].head(5)

,comparison,gse_1,gse_2,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,same_pmid,same_source_prediction,same_source_confidence,shared_pmid,title_1_full,title_2_full
0,within_mouse,GSE61031,GSE61033,0.995175,0.885714,0.987384,0.984490,False,False,Unknown,[],Dicer WT/KO MSC RNA-Seq [polyA RNA],Dicer WT/KO MSC RNA-Seq [total RNA]
1,within_mouse,GSE60556,GSE60647,0.991287,0.976923,0.979439,0.979184,False,True,High,[],From mouse to humans: detecting preventing vac...,From mouse to humans: detecting preventing vac...
2,within_human,GSE86977,GSE86985,0.986182,0.923729,0.333190,0.348964,False,True,High,[],REGION-SPECIFIC NEURAL STEM CELL LINEAGES REVE...,REGION-SPECIFIC NEURAL STEM CELL LINEAGES REVE...
3,within_mouse,GSE54286,GSE54287,0.985754,0.890909,0.999186,0.994548,False,True,High,[],The Atlas of Chromatoid Body Components [RNA-seq],The Atlas of Chromatoid Body Components [small...
4,within_mouse,GSE76458,GSE78151,0.985447,1.000000,0.966694,0.968051,False,False,Unknown,[],The transcriptome of central nervous system my...,The transcriptome of central nervous system my...
5,within_mouse,GSE60556,GSE60648,0.982267,0.969231,0.985061,0.983361,False,True,High,[],From mouse to humans: detecting preventing vac...,From mouse to humans: detecting preventing vac...
6,within_mouse,GSE45119,GSE52856,0.979568,0.963768,0.967930,0.967466,False,False,Unknown,[],Gene Expression and Exon Splicing Change Analy...,Gene Expression and Exon Splicing Change Analy...
7,within_mouse,GSE75431,GSE93179,0.977374,0.090909,0.993499,0.962343,False,False,Unknown,[],Sorted cells_PS2APP brains_7/13mo,Tau-P301L sorted cell types
8,within_mouse,GSE75431,GSE93180,0.977374,0.090909,0.993499,0.962343,False,False,Unknown,[],Sorted cells_PS2APP brains_7/13mo,Tau-P301S sorted cell types
9,within_mouse,GSE60647,GSE60648,0.973708,0.969231,0.977591,0.976705,False,True,High,[],From mouse to humans: detecting preventing vac...,From mouse to humans: detecting preventing vac...


In [ ]:
# high-high but not PMID
same_pmid_not_too_similar = (
    analysis_pairs[
        (analysis_pairs["same_pmid"]) &
        (analysis_pairs["edit_cui_pattern"] != "edit_high_cui_high_too_similar")
    ]
    .sort_values("cui_tfidf_cosine", ascending=False)
    .reset_index(drop=True)
)

same_pmid_not_too_similar[[
    "comparison",
    "gse_1",
    "gse_2",
    "edit_cui_pattern",
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim",
    "same_pmid",
    "same_source_prediction",
    "same_source_confidence",
    "shared_pmid",
    "title_1_full",
    "title_2_full"
]].head(5)

,comparison,gse_1,gse_2,edit_cui_pattern,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,same_pmid,same_source_prediction,same_source_confidence,shared_pmid,title_1_full,title_2_full
0,within_mouse,GSE95434,GSE95445,edit_low_cui_high_bio_concept_match,0.965987,0.845455,0.873294,0.868590,True,True,Very High,[29158510],Single cell RNA-seq of 346 epithelial cells fr...,Single cell RNA-seq of 444 epithelial cells fr...
1,within_human,GSE100183,GSE69876,edit_low_cui_high_bio_concept_match,0.953409,0.828571,0.852941,0.848397,True,True,Very High,[29028833],MYCL and EP400 are required for Max and MCPyV ...,EP400 is required for Max and MCPyV mediated g...
2,within_mouse,GSE93379,GSE93380,edit_low_cui_high_bio_concept_match,0.936495,0.409836,0.918452,0.879618,True,True,Very High,[28323953],Gene expression profile (RNA-seq) of hypophyse...,Gene expression profiling of Male and Female a...
3,within_mouse,GSE64099,GSE65747,edit_low_cui_high_bio_concept_match,0.931181,0.178947,0.880597,0.820163,True,True,High,[26981392],Transcriptome profiling for genes transcriptio...,Genome-wide binding and mechanistic analyses o...
4,within_mouse,GSE93380,GSE93381,edit_low_cui_high_bio_concept_match,0.924799,0.329114,0.918452,0.862391,True,True,Very High,[28323953],Gene expression profiling of Male and Female a...,Gene expression profiling (RNA-seq with partia...
5,within_mouse,GSE76853,GSE90855,edit_low_cui_high_bio_concept_match,0.918928,0.818182,0.894988,0.888889,True,True,Very High,[28562591],Acetyl-CoA metabolism by ACSS2 regulates neuro...,Acetyl CoA metabolism by ACSS2 regulates neuro...
6,within_mouse,GSE76853,GSE90854,edit_low_cui_high_bio_concept_match,0.918928,0.810811,0.894988,0.888240,True,True,Very High,[28562591],Acetyl-CoA metabolism by ACSS2 regulates neuro...,Acetyl CoA metabolism by ACSS2 regulates neuro...
7,within_human,GSE53695,GSE53697,edit_low_cui_high_bio_concept_match,0.915688,0.708333,0.818750,0.793269,True,True,Very High,[26894958],nELAVL HITS-CLIP in Alzheimer's Disease patients,RNAseq in Alzheimer's Disease patients
8,within_mouse,GSE96679,GSE96680,edit_low_cui_high_bio_concept_match,0.914282,0.578313,0.797688,0.727626,True,True,Very High,[29153407],Temperature adaptation effects on BAT metaboli...,Temperature adaptation effects on BAT metabolism
9,within_mouse,GSE95436,GSE95448,edit_low_cui_high_bio_concept_match,0.900093,0.811881,0.814145,0.814085,True,True,Very High,[29158510],Single cell RNA-seq of 278 epithelial cells fr...,Single cell RNA-seq of 312 basal epithelial ce...


In [23]:
# main groups for manual analysis
bio_concept_matches = (
    analysis_pairs[
        analysis_pairs["edit_cui_pattern"] == "edit_low_cui_high_bio_concept_match"
    ]
    .sort_values("cui_tfidf_cosine", ascending=False)
    .reset_index(drop=True)
)

possible_cui_misses = (
    analysis_pairs[
        analysis_pairs["edit_cui_pattern"] == "edit_high_cui_low_possible_cui_miss"
    ]
    .sort_values("text_damerau_sim", ascending=False)
    .reset_index(drop=True)
)

not_matching_pairs = (
    analysis_pairs[
        analysis_pairs["edit_cui_pattern"] == "edit_low_cui_low_not_matching"
    ]
    .reset_index(drop=True)
)

bio_concept_matches[[
    "comparison",
    "gse_1",
    "gse_2",
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim",
    "same_pmid",
    "same_source_prediction",
    "same_source_confidence",
    "title_1_full",
    "title_2_full"
]].head(30)

,comparison,gse_1,gse_2,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,same_pmid,same_source_prediction,same_source_confidence,title_1_full,title_2_full
0,within_mouse,GSE80166,GSE81095,0.980661,0.656566,0.940048,0.885880,False,False,Unknown,RNA Sequencing Data in differentiating mouse e...,RNA Sequencing Data in differentiating mouse e...
1,within_mouse,GSE95434,GSE95445,0.965987,0.845455,0.873294,0.868590,True,True,Very High,Single cell RNA-seq of 346 epithelial cells fr...,Single cell RNA-seq of 444 epithelial cells fr...
2,within_human,GSE100183,GSE69876,0.953409,0.828571,0.852941,0.848397,True,True,Very High,MYCL and EP400 are required for Max and MCPyV ...,EP400 is required for Max and MCPyV mediated g...
3,within_mouse,GSE93379,GSE93380,0.936495,0.409836,0.918452,0.879618,True,True,Very High,Gene expression profile (RNA-seq) of hypophyse...,Gene expression profiling of Male and Female a...
4,within_mouse,GSE64099,GSE65747,0.931181,0.178947,0.880597,0.820163,True,True,High,Transcriptome profiling for genes transcriptio...,Genome-wide binding and mechanistic analyses o...
5,within_human,GSE41476,GSE80388,0.930102,0.372549,0.290780,0.316062,False,False,Unknown,RNA-seq for gastric cancer and normal tissues/...,Gastric cancer RNA-seq
6,within_human,GSE66036,GSE70150,0.929011,0.483333,0.569378,0.553911,False,False,Unknown,Genome wide analysis of androgen-regulated tra...,Time course of androgen-regulated transcripts ...
7,within_mouse,GSE93380,GSE93381,0.924799,0.329114,0.918452,0.862391,True,True,Very High,Gene expression profiling of Male and Female a...,Gene expression profiling (RNA-seq with partia...
8,within_mouse,GSE76853,GSE90855,0.918928,0.818182,0.894988,0.888889,True,True,Very High,Acetyl-CoA metabolism by ACSS2 regulates neuro...,Acetyl CoA metabolism by ACSS2 regulates neuro...
9,within_mouse,GSE76853,GSE90854,0.918928,0.810811,0.894988,0.888240,True,True,Very High,Acetyl-CoA metabolism by ACSS2 regulates neuro...,Acetyl CoA metabolism by ACSS2 regulates neuro...


In [ ]:
analysis_pairs.to_csv(
    "similarity-results/cui_tfidf_edit_distance_pmid_analysis.csv",
    index=False
)

bio_concept_matches.to_csv(
    "similarity-results/edit_low_cui_high_bio_concept_matches.csv",
    index=False
)

too_similar_pairs.to_csv(
    "similarity-results/edit_high_cui_high_too_similar.csv",
    index=False
)

possible_cui_misses.to_csv(
    "similarity-results/edit_high_cui_low_possible_cui_misses_.csv",
    index=False
)

too_similar_not_same_pmid.to_csv(
    "similarity-results/too_similar_not_same_pmid.csv",
    index=False
)

same_pmid_not_too_similar.to_csv(
    "similarity-results/same_pmid_not_too_similar.csv",
    index=False
)

print("Saved edit-distance + CUI-TF-IDF analysis outputs.")

Saved edit-distance + CUI-TF-IDF analysis outputs.


## 11. Check group counts

In [25]:
# count and percentage of the four edit/CUI groups

pattern_summary = (
    analysis_pairs["edit_cui_pattern"]
    .value_counts()
    .rename_axis("edit_cui_pattern")
    .reset_index(name="n_pairs")
)

pattern_summary["percentage"] = (
    pattern_summary["n_pairs"] / len(analysis_pairs) * 100
)

pattern_summary

,edit_cui_pattern,n_pairs,percentage
0,edit_low_cui_low_not_matching,1884,74.466403
1,edit_low_cui_high_bio_concept_match,506,20.000000
2,edit_high_cui_high_too_similar,127,5.019763
3,edit_high_cui_low_possible_cui_miss,13,0.513834


In [26]:
pattern_summary_by_comparison = (
    analysis_pairs
    .groupby(["comparison", "edit_cui_pattern"])
    .size()
    .reset_index(name="n_pairs")
)

pattern_summary_by_comparison["percentage_within_comparison"] = (
    pattern_summary_by_comparison["n_pairs"] /
    pattern_summary_by_comparison.groupby("comparison")["n_pairs"].transform("sum") *
    100
).round(2)

display(pattern_summary_by_comparison)

,comparison,edit_cui_pattern,n_pairs,percentage_within_comparison
0,human_mouse,edit_high_cui_high_too_similar,9,0.94
1,human_mouse,edit_high_cui_low_possible_cui_miss,2,0.21
2,human_mouse,edit_low_cui_high_bio_concept_match,103,10.76
3,human_mouse,edit_low_cui_low_not_matching,843,88.09
4,within_human,edit_high_cui_high_too_similar,35,4.34
5,within_human,edit_high_cui_low_possible_cui_miss,3,0.37
6,within_human,edit_low_cui_high_bio_concept_match,160,19.83
7,within_human,edit_low_cui_low_not_matching,609,75.46
8,within_mouse,edit_high_cui_high_too_similar,83,10.84
9,within_mouse,edit_high_cui_low_possible_cui_miss,8,1.04
